# Event Match Report


In [ ]:
# Parameters are injected by smk/scripts/_4_event_match_notebook.py.
EVENT = None
DIAGNOSTICS_FP_L = []
STATUS_FP_L = []
OUTPUT_DIR = None


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

plt.rcParams['figure.facecolor'] = 'white'


In [ ]:
def _read_raster(fp):
    """Read a raster into a float array and minimal geospatial metadata."""
    with rasterio.open(fp) as ds:
        arr = ds.read(masked=True).astype('float32').filled(np.nan)
        meta_d = {
            'height': ds.height,
            'width': ds.width,
            'count': ds.count,
            'transform': ds.transform,
            'crs': ds.crs,
        }
    return arr, meta_d


def _normalize_band(band_arr, low=2, high=98):
    """Normalize one band for plotting."""
    valid_bx = np.isfinite(band_arr)
    out_arr = np.full(band_arr.shape, np.nan, dtype='float32')
    if valid_bx.sum() == 0:
        return out_arr
    low_v = np.nanpercentile(band_arr[valid_bx], low)
    high_v = np.nanpercentile(band_arr[valid_bx], high)
    if (not np.isfinite(low_v)) or (not np.isfinite(high_v)) or high_v <= low_v:
        out_arr[valid_bx] = 0.0
        return out_arr
    band_arr = np.clip(band_arr, low_v, high_v)
    out_arr[valid_bx] = (band_arr[valid_bx] - low_v) / (high_v - low_v)
    return out_arr


def _rgb_plot_arr(arr):
    """Build a visible RGB array from a PlanetScope-like raster."""
    if arr.shape[0] >= 3:
        rgb_arr = np.stack([_normalize_band(arr[2]), _normalize_band(arr[1]), _normalize_band(arr[0])], axis=-1)
    elif arr.shape[0] == 1:
        gray_arr = _normalize_band(arr[0])
        rgb_arr = np.stack([gray_arr, gray_arr, gray_arr], axis=-1)
    else:
        rgb_arr = np.stack([_normalize_band(arr[i]) for i in range(min(3, arr.shape[0]))], axis=-1)
    return np.nan_to_num(rgb_arr, nan=0.0)


def _warp_to_reference_grid(src_arr, src_meta_d, ref_meta_d):
    """Warp a candidate raster onto the reference raster grid for aligned plotting."""
    dst_arr = np.full((src_arr.shape[0], ref_meta_d['height'], ref_meta_d['width']), np.nan, dtype='float32')
    for band_i in range(src_arr.shape[0]):
        reproject(
            source=src_arr[band_i],
            destination=dst_arr[band_i],
            src_transform=src_meta_d['transform'],
            src_crs=src_meta_d['crs'],
            src_nodata=np.nan,
            dst_transform=ref_meta_d['transform'],
            dst_crs=ref_meta_d['crs'],
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return dst_arr


def _fmt_ts(value):
    """Format a timestamp-like value for plot titles."""
    if value is None or pd.isna(value):
        return 'NA'
    return pd.Timestamp(value).strftime('%Y-%m-%d %H:%M')


def _attempt_sort_key(attempt_d):
    """Sort attempts by scene datetime then item id."""
    acquired = pd.Timestamp(attempt_d.get('acquired')) if attempt_d.get('acquired') else pd.Timestamp.max.tz_localize('UTC')
    return acquired, str(attempt_d.get('item_id'))


def _plot_chip(match_d):
    """Plot one chip with reference over its matched attempt or far right when unmatched."""
    chip_context = match_d['chip_context']
    attempt_l = [d for d in sorted(match_d.get('attempts', []), key=_attempt_sort_key) if d.get('raster_fp')]
    ncols = max(len(attempt_l), 1)
    matched_idx = None
    for idx, attempt_d in enumerate(attempt_l):
        if (attempt_d.get('compare_metrics') or {}).get('matched', False):
            matched_idx = idx
            break
    ref_col = matched_idx if matched_idx is not None else ncols - 1
    reference_fp = Path(match_d['reference_fp'])
    ref_arr, ref_meta_d = _read_raster(reference_fp)
    fig, ax_ar = plt.subplots(2, ncols, figsize=(max(4.2 * ncols, 5.0), 7.0), squeeze=False, constrained_layout=True)
    fig.suptitle(f"{chip_context['event']} | {chip_context['chip_id']}", fontsize=12)
    for ax in ax_ar.ravel():
        ax.set_axis_off()
    ax_ar[0, ref_col].imshow(_rgb_plot_arr(ref_arr), interpolation='nearest', aspect='equal')
    ax_ar[0, ref_col].set_title(f"source\n{_fmt_ts(chip_context.get('reference_datetime'))}", fontsize=9)
    if not attempt_l:
        ax_ar[1, 0].text(0.5, 0.5, 'no attempted rasters', ha='center', va='center', transform=ax_ar[1, 0].transAxes)
        plt.show()
        return fig
    for idx, attempt_d in enumerate(attempt_l):
        fetch_fp = Path(attempt_d['raster_fp'])
        if not fetch_fp.exists():
            ax_ar[1, idx].text(0.5, 0.5, 'missing raster', ha='center', va='center', transform=ax_ar[1, idx].transAxes)
            continue
        fetch_arr, fetch_meta_d = _read_raster(fetch_fp)
        fetch_arr = _warp_to_reference_grid(fetch_arr, fetch_meta_d, ref_meta_d)
        metrics_d = attempt_d.get('compare_metrics') or {}
        matched = bool(metrics_d.get('matched', False))
        title = f"{attempt_d.get('item_id')}\n{_fmt_ts(attempt_d.get('acquired'))}\nmatched={matched} rank={attempt_d.get('match_candidate_rank')}"
        ax_ar[1, idx].imshow(_rgb_plot_arr(fetch_arr), interpolation='nearest', aspect='equal')
        ax_ar[1, idx].set_title(title, fontsize=8)
        for spine in ax_ar[1, idx].spines.values():
            spine.set_visible(True)
            spine.set_linewidth(2.0 if matched else 0.8)
            spine.set_edgecolor('#2ca02c' if matched else '#777777')
    plt.show()
    return fig


In [ ]:
match_d_l = [json.loads(Path(fp).read_text()) for fp in DIAGNOSTICS_FP_L]
status_d_l = [json.loads(Path(fp).read_text()) for fp in STATUS_FP_L]
status_df = pd.DataFrame(status_d_l)
display(status_df[['event', 'chip_id', 'status', 'matched', 'attempted_candidates']].sort_values(['event', 'chip_id']))


In [ ]:
for match_d in sorted(match_d_l, key=lambda d: d['chip_context']['chip_id']):
    _plot_chip(match_d)
